# Classificação Binária de Recomendação - B2W Reviews

Este notebook adapta o tutorial oficial do TensorFlow [Classificação de texto com avaliações de filmes](https://www.tensorflow.org/tutorials/keras/text_classification?hl=pt-br) para o conjunto de dados **B2W-Reviews01**.

O objetivo é prever, a partir do texto de uma avaliação escrita por um cliente, se ele **recomendaria o produto a um amigo** (coluna `recommend_to_a_friend`, com valores `Yes` ou `No`). Trata-se, portanto, de um problema de classificação binária.

A estrutura segue a mesma lógica do tutorial: representamos cada avaliação como uma sequência de inteiros, aplicamos *padding* para igualar os tamanhos e treinamos um modelo simples baseado em `Embedding`, `GlobalAveragePooling1D` e camadas densas. Ao final, o modelo é salvo com `pickle` e uma função recebe novas avaliações e devolve a classificação.

## 1. Importação das bibliotecas

Como no tutorial, importamos `tensorflow`, `keras` e `numpy`. Como o dataset do B2W vem em arquivo CSV (e não pré-empacotado dentro do Keras como o IMDB), também é necessário o `pandas` para ler o arquivo. A versão do TensorFlow é exibida apenas para conferência.

In [ ]:
import tensorflow as tf
from tensorflow import keras

import numpy as np
import pandas as pd

print(tf.__version__)

## 2. Carregamento do dataset B2W

O arquivo `B2W-Reviews01.csv` contém milhares de avaliações de produtos em português. Para o desafio, interessam apenas duas colunas: `review_text`, com o texto da avaliação, e `recommend_to_a_friend`, com a resposta (`Yes` ou `No`) à pergunta se o cliente recomendaria o produto a um amigo. Linhas com valores faltantes nessas colunas são descartadas, pois não podem ser usadas no aprendizado supervisionado.

In [ ]:
# Caminho do dataset (ajuste se estiver rodando no Google Colab)
DATA_PATH = "../data/B2W-Reviews.csv"

df = pd.read_csv(DATA_PATH)
df = df[["review_text", "recommend_to_a_friend"]].dropna()
df.head()

## 3. Exploração inicial dos dados

Antes de treinar o modelo, observamos o total de avaliações disponíveis e a distribuição das classes. Um desbalanceamento muito forte (uma classe muito mais frequente do que a outra) pode prejudicar o aprendizado, pois o modelo tende a prever sempre a classe majoritária.

In [ ]:
print("Total de avaliações:", len(df))
print("\nDistribuição das classes:")
print(df["recommend_to_a_friend"].value_counts())

## 4. Codificação dos rótulos

Como o modelo trabalha com números, a coluna `recommend_to_a_friend` é convertida em valores binários: `1` para `Yes` (recomenda) e `0` para `No` (não recomenda).

In [ ]:
df["label"] = df["recommend_to_a_friend"].map({"Yes": 1, "No": 0})
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)
df[["review_text", "label"]].head()

## 5. Separação em treino e teste

Como o B2W vem em um único arquivo, separamos manualmente cerca de 80% das avaliações para treino e 20% para teste. Antes disso, o dataframe é embaralhado para evitar viés de ordem temporal.

In [ ]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

n_train = int(0.8 * len(df))

train_texts = df["review_text"].iloc[:n_train].values
test_texts = df["review_text"].iloc[n_train:].values

train_labels = df["label"].iloc[:n_train].values
test_labels = df["label"].iloc[n_train:].values

print("Training entries: {}, labels: {}".format(len(train_texts), len(train_labels)))

## 6. Construção do vocabulário e tokenização

Diferente do IMDB, o B2W não vem pré-tokenizado. Para reproduzir o formato esperado pelo modelo (sequências de inteiros), usamos a classe `Tokenizer` do Keras. Ela aprende as 10.000 palavras mais frequentes do conjunto de treino (`num_words=10000`) e converte cada palavra em um índice inteiro. Palavras desconhecidas são representadas pelo token `<UNK>`.

In [ ]:
vocab_size = 10000

tokenizer = keras.preprocessing.text.Tokenizer(num_words=vocab_size, oov_token="<UNK>")
tokenizer.fit_on_texts(train_texts)

train_data = tokenizer.texts_to_sequences(train_texts)
test_data = tokenizer.texts_to_sequences(test_texts)

print("Tamanho do vocabulário aprendido:", min(vocab_size, len(tokenizer.word_index) + 1))
print("\nPrimeira avaliação como sequência de inteiros:")
print(train_data[0][:30])

## 7. Aplicação de *padding*

As avaliações têm comprimentos variados, mas a rede neural exige entradas de tamanho fixo. Aplicamos `pad_sequences` com `maxlen=256`, completando as sequências menores com zeros ao final e cortando as maiores. O valor `0` é reservado para preenchimento (como o `<PAD>` do tutorial).

In [ ]:
maxlen = 256

train_data = keras.preprocessing.sequence.pad_sequences(
    train_data, value=0, padding="post", maxlen=maxlen
)

test_data = keras.preprocessing.sequence.pad_sequences(
    test_data, value=0, padding="post", maxlen=maxlen
)

print("Formato do conjunto de treino após padding:", train_data.shape)
print("Formato do conjunto de teste após padding:", test_data.shape)

## 8. Construção do modelo

Reaproveitamos exatamente a arquitetura proposta pelo tutorial:

1. **Embedding**: associa cada índice de palavra a um vetor denso de dimensão 16.
2. **GlobalAveragePooling1D**: tira a média dos vetores ao longo da sequência, gerando um único vetor por avaliação.
3. **Dense(16, relu)**: camada totalmente conectada com ativação ReLU.
4. **Dense(1, sigmoid)**: camada de saída que devolve a probabilidade entre 0 e 1 de o cliente recomendar o produto.

In [ ]:
model = keras.Sequential()
model.add(keras.layers.Embedding(vocab_size, 16))
model.add(keras.layers.GlobalAveragePooling1D())
model.add(keras.layers.Dense(16, activation="relu"))
model.add(keras.layers.Dense(1, activation="sigmoid"))

model.summary()

## 9. Compilação do modelo

Os mesmos componentes usados no tutorial atendem ao nosso problema, que também é binário: otimizador `adam`, perda `binary_crossentropy` e métrica `accuracy`.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

## 10. Conjunto de validação

Seguindo a abordagem do tutorial, separamos 10.000 exemplos do treino como conjunto de validação. Esses dados serão usados para monitorar o aprendizado a cada época, sem comprometer o conjunto de teste.

In [ ]:
x_val = train_data[:10000]
partial_x_train = train_data[10000:]

y_val = train_labels[:10000]
partial_y_train = train_labels[10000:]

print("Validação:", x_val.shape)
print("Treino efetivo:", partial_x_train.shape)

## 11. Treinamento do modelo

O modelo é treinado por 40 épocas com lotes de 512 exemplos. A cada época, o desempenho no conjunto de validação é avaliado, permitindo identificar problemas de generalização ao longo do tempo.

In [ ]:
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=40,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1,
)

## 12. Avaliação no conjunto de teste

O conjunto de teste não foi usado em nenhuma etapa anterior. A avaliação nele fornece uma estimativa honesta do desempenho do modelo em avaliações novas.

In [ ]:
results = model.evaluate(test_data, test_labels, verbose=2)

print(results)

## 13. Histórico de treinamento

O objeto `history` armazena perda e métricas a cada época. Esses dados serão usados para construir os gráficos a seguir.

In [ ]:
history_dict = history.history
history_dict.keys()

## 14. Gráfico de perda

Comparar a perda de treino com a de validação ao longo das épocas ajuda a identificar problemas de generalização. Se a perda de validação começar a subir enquanto a de treino continua caindo, é sinal de *overfitting*.

In [ ]:
import matplotlib.pyplot as plt

acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
loss = history_dict["loss"]
val_loss = history_dict["val_loss"]

epochs = range(1, len(acc) + 1)

plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()

## 15. Gráfico de acurácia

Da mesma forma, comparar a acurácia de treino e validação mostra a evolução do modelo em termos de proporção de acertos.

In [ ]:
plt.clf()

plt.plot(epochs, acc, "bo", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

## 16. Exportação do modelo com `pickle`

O enunciado pede que o modelo treinado seja salvo com `pickle`. Como modelos Keras não são serializáveis diretamente por `pickle` de forma confiável, salvamos o modelo em formato Keras nativo (`.keras`) e armazenamos com `pickle` um *bundle* contendo o caminho do modelo, o tokenizer aprendido (necessário para preparar novas entradas), o comprimento máximo das sequências e o mapeamento dos rótulos.

In [ ]:
import os
import pickle

MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

keras_model_path = os.path.join(MODELS_DIR, "b2w_model.keras")
pickle_path = os.path.join(MODELS_DIR, "b2w_model.pkl")

model.save(keras_model_path)

bundle = {
    "keras_model_path": keras_model_path,
    "tokenizer": tokenizer,
    "maxlen": maxlen,
    "label_mapping": {0: "No", 1: "Yes"},
}

with open(pickle_path, "wb") as f:
    pickle.dump(bundle, f)

print("Modelo salvo em:", keras_model_path)
print("Bundle pickle salvo em:", pickle_path)

## 17. Função de classificação para novos textos

A função `classificar_recomendacao` recebe um texto novo e o caminho do `pickle` salvo. Internamente, ela carrega o modelo e o tokenizer, transforma o texto em uma sequência de inteiros, aplica o *padding* até `maxlen` e usa o modelo para prever a probabilidade de recomendação. O resultado é um dicionário com o texto, a probabilidade estimada e a classe predita (`Yes` ou `No`).

In [ ]:
def classificar_recomendacao(texto, pickle_path="../models/b2w_model.pkl"):
    with open(pickle_path, "rb") as f:
        b = pickle.load(f)

    modelo = keras.models.load_model(b["keras_model_path"])
    tok = b["tokenizer"]
    maxlen_local = b["maxlen"]

    sequencia = tok.texts_to_sequences([texto])
    sequencia = keras.preprocessing.sequence.pad_sequences(
        sequencia, value=0, padding="post", maxlen=maxlen_local
    )

    probabilidade = float(modelo.predict(sequencia, verbose=0)[0][0])
    classe = b["label_mapping"][1 if probabilidade >= 0.5 else 0]

    return {
        "texto": texto,
        "probabilidade_recomendar": probabilidade,
        "classe_predita": classe,
    }

In [ ]:
exemplos = [
    "Adorei o produto, entrega rápida e qualidade excelente. Super recomendo!",
    "Produto chegou quebrado, atendimento péssimo, não comprem nesta loja.",
    "Atendeu minhas expectativas, vale muito a pena pelo preço.",
    "Não funcionou direito, tive que devolver. Fiquei muito decepcionado.",
]

for texto in exemplos:
    print(classificar_recomendacao(texto))

## 18. Considerações finais

Este notebook adaptou o tutorial de classificação de texto do TensorFlow para um cenário em português, utilizando o dataset B2W-Reviews01 e a coluna `recommend_to_a_friend` como alvo binário. As principais adaptações em relação ao tutorial original foram:

- Leitura do dataset com `pandas`, já que o B2W vem em um arquivo CSV.
- Codificação manual dos rótulos `Yes`/`No` em `1`/`0`.
- Tokenização com `keras.preprocessing.text.Tokenizer`, no lugar do dicionário pré-construído usado pelo IMDB.
- Exportação do modelo com `pickle` e criação da função `classificar_recomendacao` para inferência em novos textos.

Como evolução, seria interessante explorar técnicas que lidem melhor com o desbalanceamento de classes (já que a maioria dos clientes recomenda o produto) e o uso de *embeddings* pré-treinados em português.